## Importing Libraries
We imported the necessary libraries for data handling, preprocessing, model building, and evaluation, including Pandas, scikit-learn modules, and Random Forest Classifier.


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

## Loading Datasets
We loaded two datasets, `quotes_won.csv` and `quotes_lost.csv`, into separate Pandas DataFrames.


In [4]:
# Load datasets
file_won = "../../Dataset/quotes_won.csv"
file_lost = "../../Dataset/quotes_lost.csv"
df_won = pd.read_csv(file_won)
df_lost = pd.read_csv(file_lost)

## Combining Datasets
We combined the "won" and "lost" quotes into a single DataFrame, resetting the index.


In [5]:
# Concatenate datasets
df_combined = pd.concat([df_won, df_lost], ignore_index=True)


## Dropping Unnecessary Columns
We dropped columns that were not useful for modeling, specifically `QuoteID`, `OrderNumber`, and `ConversionDate`.


In [8]:
# Drop unnecessary columns
drop_cols = ["QuoteID", "OrderNumber", "ConversionDate"]
df_clean = df_combined.drop(columns=drop_cols)

## Converting Date Columns
We converted the `QuoteDate` and `ExpirationDate` columns into proper datetime format to facilitate feature engineering.


In [9]:
# Convert dates to datetime format
date_cols = ["QuoteDate", "ExpirationDate"]
for col in date_cols:
    df_clean[col] = pd.to_datetime(df_clean[col], errors="coerce")

## Creating Date-Related Feature
We created a new feature `QuoteToExpireDays` representing the number of days between the quote date and expiration date.


In [10]:

# Create new date-related features
df_clean["QuoteToExpireDays"] = (df_clean["ExpirationDate"] - df_clean["QuoteDate"]).dt.days

## Dropping Original Date Columns
We dropped the original `QuoteDate` and `ExpirationDate` columns after extracting the necessary information.


In [11]:
# Drop original date columns
df_clean = df_clean.drop(columns=date_cols)

## Handling Missing Numerical Values
We filled missing values in numerical columns with their respective median values.


In [13]:
# Fill missing values in numerical columns with the median
num_cols = df_clean.select_dtypes(include=["number"]).columns
df_clean[num_cols] = df_clean[num_cols].fillna(df_clean[num_cols].median())

## Preparing for Encoding Categorical Variables
We identified all categorical columns that required encoding and initialized a dictionary to store label encoders.


In [14]:
# Encode categorical variables
cat_cols = df_clean.select_dtypes(include=["object"]).columns.tolist()
label_encoders = {}

## Encoding Categorical Variables
We applied label encoding to all categorical columns and saved the encoders for potential future use.


In [15]:
for col in cat_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col])
    label_encoders[col] = le  # Store encoder for future use

## Defining Features and Target
We separated the dataset into features (`X`) and target variable (`y`), where `QuoteStatus` was used as the target.


In [17]:
# Define features and target
X = df_clean.drop(columns=["QuoteStatus"])
y = df_clean["QuoteStatus"]


## Splitting Data into Training and Testing Sets
We split the data into training and testing sets, ensuring a stratified split to maintain class distribution.


In [18]:
# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## Training Random Forest Model
We initialized a Random Forest Classifier with specific hyperparameters and trained it on the training set.


In [20]:
# Initialize and train Random Forest model
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

RandomForestClassifier(max_depth=10, min_samples_leaf=5, min_samples_split=10,
                       n_jobs=-1, random_state=42)

## Making Predictions
We used the trained Random Forest model to predict outcomes on the testing set.


In [21]:
# Make predictions
y_pred = rf_model.predict(X_test)

## Evaluating Model Performance
We evaluated the model’s performance by calculating the accuracy and generating a classification report.


In [22]:
# Evaluate model performance
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

## Printing Results
We printed the accuracy score and the detailed classification report for the model predictions.


In [24]:
# Print results
print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:\n", report)

Accuracy: 0.8678
Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.58      0.73     90752
           1       0.84      1.00      0.91    201623

    accuracy                           0.87    292375
   macro avg       0.92      0.79      0.82    292375
weighted avg       0.89      0.87      0.86    292375

